# 考古墓葬数据库 - OCR Pipeline

处理考古报告 PDF → 提取墓葬信息 → 写入 SQLite

## 架构
- **Track B**: OCR 管线（本 notebook）
- **Track A**: CSV 导入 + GIS（后续）

In [ ]:
import os, sys, sqlite3, json
from pathlib import Path

# 项目路径
PROJECT = Path('/root/xiaohanchen/projects/vps-llm-kaogu')
os.chdir(PROJECT)

# 数据库
DB = PROJECT / 'kaogu.db'
conn = sqlite3.connect(DB)
conn.row_factory = sqlite3.Row

print(f'数据库: {DB}')
print(f'大小: {DB.stat().st_size / 1024:.1f} KB')

In [ ]:
# 检查当前数据库状态
tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
print('表:', [t[0] for t in tables])

# 各表记录数
for t in tables:
    count = conn.execute(f'SELECT count(*) FROM {t[0]}').fetchone()[0]
    if count > 0:
        print(f'  {t[0]}: {count} 条')

In [ ]:
# 列出 VPS 上的 PDF
pdfs = sorted((PROJECT / 'data' / 'pdfs').glob('*.pdf'))
print(f'PDF 数量: {len(pdfs)}')
for p in pdfs:
    size_mb = p.stat().st_size / 1024 / 1024
    print(f'  {p.name} ({size_mb:.1f} MB)')

In [ ]:
# 查看处理进度
progress = conn.execute('SELECT * FROM report_progress').fetchall()
print('处理进度:')
for row in progress:
    print(f'  {row["title"]}: {row["ocr_status"]} — {row["tomb_count"]} 墓, {row["artifact_count"]} 器物')

In [ ]:
# 运行 OCR 管线（单个 PDF 测试）
# 取消注释运行:
# !python scripts/ocr_pipeline.py "data/pdfs/满城汉墓_上册.pdf" --max-pages 50

In [ ]:
# 批量处理所有 PDF
# 取消注释运行:
# !python scripts/ocr_pipeline.py --batch data/pdfs/

In [ ]:
# 查询墓葬概览
tombs = conn.execute('SELECT * FROM tomb_overview LIMIT 20').fetchall()
for t in tombs:
    print(f'{t["tomb_number"]}: {t["dynasty"] or "?"} | {t["structure"] or "?"} | '
          f'{t["length"] or "?"}x{t["width"] or "?"}x{t["depth"] or "?"}m | '
          f'{t["artifact_count"]} 件器物')

In [ ]:
# 器物统计
stats = conn.execute('SELECT * FROM artifact_stats ORDER BY count DESC LIMIT 20').fetchall()
print('器物统计:')
for s in stats:
    print(f'  {s["material"]}/{s["vessel_type"]}: {s["count"]} 件 ({s["tomb_count"]} 墓)')

In [ ]:
# 导出 GeoJSON（用于 GIS 可视化）
geojson = {
    'type': 'FeatureCollection',
    'features': []
}

tombs = conn.execute('''
    SELECT t.tomb_number, t.dynasty, t.structure, t.length, t.width, t.depth,
           s.name as site_name, s.province, s.longitude, s.latitude
    FROM tombs t
    LEFT JOIN sites s ON t.site_id = s.id
    WHERE s.longitude IS NOT NULL
''').fetchall()

for t in tombs:
    feature = {
        'type': 'Feature',
        'geometry': {
            'type': 'Point',
            'coordinates': [t['longitude'], t['latitude']]
        },
        'properties': {
            'tomb_number': t['tomb_number'],
            'dynasty': t['dynasty'],
            'structure': t['structure'],
            'dimensions': f"{t['length']}x{t['width']}x{t['depth']}m",
            'site': t['site_name'],
            'province': t['province'],
        }
    }
    geojson['features'].append(feature)

output = PROJECT / 'data' / 'tombs.geojson'
with open(output, 'w') as f:
    json.dump(geojson, f, ensure_ascii=False, indent=2)

print(f'导出 {len(geojson["features"])} 个墓葬到 {output}')

In [ ]:
conn.close()
print('Done')